In [1]:
import os
import urllib.request
import sys

# Define the local Hadoop directory path
hadoop_home = os.path.join(os.getcwd(), "hadoop")
bin_dir = os.path.join(hadoop_home, "bin")

# Create directories if they do not exist
os.makedirs(bin_dir, exist_ok=True)

# URLs for Hadoop binaries required by Spark on Windows
winutils_url = "https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.3.5/bin/winutils.exe"
hadoop_dll_url = "https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.3.5/bin/hadoop.dll"

winutils_path = os.path.join(bin_dir, "winutils.exe")
hadoop_dll_path = os.path.join(bin_dir, "hadoop.dll")

# Download winutils.exe if it does not exist
if not os.path.exists(winutils_path):
    print("Downloading winutils.exe...")
    urllib.request.urlretrieve(winutils_url, winutils_path)

# Download hadoop.dll if it does not exist
if not os.path.exists(hadoop_dll_path):
    print("Downloading hadoop.dll...")
    urllib.request.urlretrieve(hadoop_dll_url, hadoop_dll_path)

# CRITICAL: Set environment variables for the current Python session
os.environ["HADOOP_HOME"] = hadoop_home
os.environ["hadoop.home.dir"] = hadoop_home

# Add Hadoop bin directory to the system PATH so the JVM can find hadoop.dll
os.environ["PATH"] = bin_dir + os.pathsep + os.environ.get("PATH", "")

print("Hadoop environment setup is complete. You can now start the Spark session.")

Hadoop environment setup is complete. You can now start the Spark session.


In [2]:
# Question 1: Install Spark and PySpark

import pyspark
from pyspark.sql import SparkSession

# Create a local Spark session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Zoomcamp_HW6") \
    .getOrCreate()

# Execute and print Spark version for Question 1
print("Question 1 Answer:", spark.version)

Question 1 Answer: 4.1.1


In [3]:
# Question 2: Yellow November 2025

import urllib.request
import os
import glob

# Define file name and URL
taxi_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
taxi_file = "yellow_tripdata_2025-11.parquet"

# Download the Yellow taxi data if it does not exist
if not os.path.exists(taxi_file):
    print("Downloading Yellow Taxi Data for Nov 2025...")
    urllib.request.urlretrieve(taxi_url, taxi_file)
    print("Download completed.")

# Read the November 2025 Yellow data into a Spark Dataframe
df_taxi = spark.read.parquet(taxi_file)

# Repartition the Dataframe to 4 partitions and save it to parquet
output_path = "yellow_2025_11_repartitioned"
df_taxi.repartition(4).write.mode("overwrite").parquet(output_path)

# Calculate the average size of the Parquet files created (in MB)
parquet_files = glob.glob(f"{output_path}/*.parquet")
sizes_mb = [os.path.getsize(f) / (1024 * 1024) for f in parquet_files]

if sizes_mb:
    avg_size = sum(sizes_mb) / len(sizes_mb)
    print(f"Question 2 Answer: The average size is approximately {avg_size:.2f} MB")

Question 2 Answer: The average size is approximately 24.42 MB


In [5]:
# Question 3: Count records

from pyspark.sql import functions as F

# Filter the dataframe for trips that started on November 15, 2025
# The column for pickup time in Yellow Taxi data is 'tpep_pickup_datetime'
df_nov15 = df_taxi.filter(F.to_date(df_taxi.tpep_pickup_datetime) == '2025-11-15')

# Count the number of trips
trip_count = df_nov15.count()

print(f"Question 3 Answer: The number of taxi trips on November 15th is {trip_count}")

Question 3 Answer: The number of taxi trips on November 15th is 162604


In [8]:
# Question 4: Longest trip

# Create a new column 'duration_hours' by subtracting pickup time from dropoff time
# F.unix_timestamp converts the datetime to seconds, so we divide by 3600 to get hours
df_duration = df_taxi.withColumn(
    "duration_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

# Find the maximum value in the 'duration_hours' column
max_duration = df_duration.select(F.max("duration_hours")).collect()[0][0]

print(f"Question 4 Answer: The length of the longest trip is {max_duration:.2f} hours")

Question 4 Answer: The length of the longest trip is 90.65 hours


In [9]:
# Question 6: Least frequent pickup location zone

import urllib.request
import os

# Define URL and filename for the zone lookup data
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zone_file = "taxi_zone_lookup.csv"

# Download the zone lookup data if it does not exist
if not os.path.exists(zone_file):
    print("Downloading Taxi Zone Lookup Data...")
    urllib.request.urlretrieve(zone_url, zone_file)
    print("Download completed.")

# Load the zone lookup data into a Spark DataFrame
df_zones = spark.read.option("header", "true").csv(zone_file)

# Create temporary views for both DataFrames to use Spark SQL
df_zones.createOrReplaceTempView("zones")
df_taxi.createOrReplaceTempView("yellow_taxi")

# Write an SQL query to join the tables, count pickups per zone, and find the minimum
query = """
    SELECT z.Zone, COUNT(t.PULocationID) as trip_count
    FROM yellow_taxi t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY trip_count ASC
    LIMIT 1
"""

# Execute the query and fetch the result
least_frequent_zone = spark.sql(query).collect()[0]

print(f"Question 6 Answer: The least frequent pickup location zone is '{least_frequent_zone['Zone']}' with {least_frequent_zone['trip_count']} trips.")

Download completed.
Question 6 Answer: The least frequent pickup location zone is 'Governor's Island/Ellis Island/Liberty Island' with 1 trips.
